# Synthetischer Daten-Generator für Banking-Fraud-Detection

Dieses Notebook erklärt den bereitgestellten Python-Code Schritt für Schritt.

**Ziel des Skripts:** Es erzeugt synthetische Telefontranskripte für ein Fraud-Detection-SFT-Dataset. Dabei werden **NeMo Curator**, ein lokal erreichbarer **OpenAI-kompatibler NIM-Server**, **NeMo Guardrails**, **Pydantic** und **asynchrone Verarbeitung mit `asyncio`** kombiniert.

Der Ablauf lässt sich grob so zusammenfassen:

1. Konfiguration und Abhängigkeiten laden
2. Verbindung zum lokalen LLM testen
3. Fraud- und legitime Szenarien definieren
4. Zufällige Gesprächsparameter erzeugen
5. Gespräche über das LLM generieren
6. Ausgabe als JSON bereinigen und validieren
7. Qualität mit einem zweiten LLM-Aufruf bewerten
8. Nur Datensätze mit Score ≥ 4 speichern
9. Viele Anfragen parallel ausführen


## 1. Imports

Der Code verwendet mehrere Bibliotheken mit unterschiedlichen Aufgaben:

- `asyncio`: parallele/asynchrone Verarbeitung
- `json`: Lesen und Schreiben von JSON
- `os`: Zugriff auf Environment Variables
- `random`: zufällige Auswahl von Szenarien, Namen und Tonlagen
- `re`: Bereinigung von LLM-Ausgaben mit regulären Ausdrücken
- `typing.Tuple`: Typisierung
- `pydantic`: Validierung der generierten Daten
- `openai`: Verbindung zu einem OpenAI-kompatiblen Endpoint
- `nemo_curator`: Generierung über NeMo Curator
- `nemoguardrails`: Einbindung der Guardrails


In [1]:
"""
Synthetischer Daten-Generator für Banking-Betrugserkennung (Fraud Detection SFT mit Guardrails)

Dieses Skript generiert asynchron realistische Gesprächstranskripte von Telefonaten 
zwischen Kunden und Bankmitarbeitern mithilfe von NeMo Curator, NVIDIA NIM APIs und NeMo Guardrails.

Autor:         Daniel Lohmann
Datum:         2026
Erforlgreich getestet am: 19.08.2026
"""

import asyncio
import json
import os
import random
import re
from typing import Tuple
from pydantic import BaseModel, Field, ValidationError
from openai import OpenAI
from nemo_curator import OpenAIClient
from nemo_curator.synthetic import NemotronGenerator
from nemoguardrails import LLMRails, RailsConfig
completed_counter = 0

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-23 10:04:08.087985059 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


## 2. Konfiguration

Hier werden die wichtigsten Laufzeitparameter festgelegt.

`NIM_BASE_URL` enthält die Adresse des lokalen NIM-Servers. Über `os.getenv()` kann diese Adresse außerhalb des Codes überschrieben werden.

`GEN_MODEL` ist das Modell für die Generierung. `JUDGE_MODEL` wird für die Qualitätsbewertung verwendet.

Die beiden Ausgabedateien speichern:

- die eigentlichen Transkripte
- ein Benchmark mit `id`, `label` und Qualitäts-Score

`NUM_SAMPLES = 100` bedeutet, dass bis zu 100 Generierungsaufgaben gestartet werden.

`CONCURRENCY_LIMIT = 8` begrenzt gleichzeitig laufende Aufgaben auf acht.


In [9]:
# 1. Konfiguration über Environment Variables (Lokaler NIM Server Port 8800)
NIM_BASE_URL = os.getenv("NIM_BASE_URL", "http://172.17.0.1:8800/v1")
GEN_MODEL = "meta/llama-3.1-8b-instruct"
JUDGE_MODEL = "meta/llama-3.1-8b-instruct"  # LLM-as-a-Judge Modell
OUTPUT_DATA_FILE = "/data/nemo-fraud-detection-notebooks/notebooks/01_Data_Generation/data/transcripts.jsonl"
OUTPUT_BENCHMARK_FILE = "/data/nemo-fraud-detection-notebooks/notebooks/01_Data_Generation/data/benchmark.jsonl"
NUM_SAMPLES = 5000
CONCURRENCY_LIMIT = 8

# NeMo Curator Client Setup (angepasst auf lokalen OpenAI-kompatiblen NIM Server)
base_openai_client = OpenAI(base_url=NIM_BASE_URL, api_key="not-needed")
curator_openai_client = OpenAIClient(base_openai_client)
generator = NemotronGenerator(curator_openai_client)

semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
file_lock = asyncio.Lock()

## 3. Datenbasis: Namen, Szenarien und Tonlagen

Die Listen bilden die Variationsmöglichkeiten des Generators.

Es gibt getrennte Kataloge für:

- **Fraud-Szenarien**: z. B. Identitätsmissbrauch, Umgehung von Sicherheitsprüfungen oder unberechtigte Limitänderungen.
- **Legitime Szenarien**: normale Kundenanliegen wie Daueraufträge, Kartenverlust oder Steuerbescheinigungen.
- **Tones**: unterschiedliche Gesprächsstile.
- **LENGTH_PROMPTS**: kurze, mittlere und lange Gespräche.

Dadurch kann aus einem relativ kleinen Szenarienkatalog eine deutlich vielfältigere synthetische Datenmenge erzeugt werden.


In [10]:
# 2. Erweiterter Namens-Pool (20 Namen) für maximale Varianz im Finetuning
KUNDEN_NAMEN = [
    "Herr Müller", "Frau Schmidt", "Herr Schneider", "Frau Fischer", 
    "Herr Weber", "Frau Meyer", "Herr Wagner", "Frau Becker", 
    "Herr Hoffmann", "Frau Schulz", "Herr Koch", "Frau Bauer", 
    "Herr Richter", "Frau Klein", "Herr Wolf", "Frau Schröder", 
    "Herr Neumann", "Frau Schwarz", "Herr Zimmermann", "Frau Braun"
]

# 3. Szenarien-Katalog: Fraud & Legitimate
FRAUD_SCENARIOS = [
    "Kunde fordert sofortige Kontoentsperrung, verweigert aber Sicherheitscodes wegen angeblich defektem Handy.",
    "Kunde verlangt Passwort-Reset für Online-Banking und nutzt entwendete Stammdaten, scheitert aber an Sicherheitsfrage.",
    "Kunde fordert Eilüberweisung und setzt den Bankmitarbeiter wegen angeblicher Notlage massiv unter Druck.",
    "Kunde versucht eine neue Telefonnummer / Adresse ohne PostIdent oder SMS-TAN im System hinterlegen zu lassen.",
    "Kunde fordert plötzliche Anhebung des Tageslimits für Überweisungen mit der Ausrede eines Spontankaufs.",
    "Kunde gibt vor, im Ausland bestohlen worden zu sein, und verlangt Notfall-Bargeldauszahlung ohne Ausweisdokumente.",
    "Kunde versucht, eine Fremdkarte/Zweitkarte auf eine neue Adresse bestellen zu lassen (Identity Theft).",
    "Kunde fragt gezielt Details zum Kontostand und letzten Buchungen ab, ohne die vollständige Legitimation erbringen zu können.",
    "Kunde behauptet, die 2FA-App funktioniere nicht, und versucht den Mitarbeiter zu überreden, die TAN manuell freizugeben.",
    "Kunde gibt sich als bevollmächtigter Angehöriger eines Senioren aus, hat aber keine eingetragene Vollmacht."
]

LEGITIMATE_SCENARIOS = [
    "Kunde erfragt Kontostand und die letzten Buchungen der letzten zwei Wochen.",
    "Kunde möchte einen bestehenden Dauerauftrag bezüglich Höhe und Ausführungstag ändern.",
    "Kunde erkundigt sich nach den Voraussetzungen und Zinsen für ein Festgeldkonto.",
    "Kunde hat seine PIN dreimal falsch eingegeben und bittet um Hilfe zur Freischaltung über den regulären Prozess.",
    "Kunde meldet einen ordnungsgemäßen Umzug und lässt seine Adresse nach erfolgreicher 2FA/Legitimation ändern.",
    "Kunde möchte seine verloren gegangene Debitkarte sperren lassen und eine Ersatzkarte bestellen.",
    "Kunde fragt nach Informationen zur Freischaltung des Online-Bankings für das Smartphone.",
    "Kunde versteht eine Abbuchung auf dem Kontoauszug nicht und lässt sich den Händlernamen erklären.",
    "Kunde möchte vor einem Urlaub das Limit für Kartenzahlungen im Ausland temporär anpassen.",
    "Kunde fordert eine Steuerbescheinigung für das vergangene Jahr an."
]

TONES = [
    "sehr drängend, hektisch und autoritär",
    "verwirrt, unsicher und zögerlich",
    "extrem freundlich, charmant und ablenkend",
    "panisch, emotional aufgeladen und wütend",
    "sachlich, professionell und bestimmt",
    "ungeduldig und leicht genervt"
]

LENGTH_PROMPTS: list[Tuple[str, int]] = [
    ("kurz (ca. 4-6 Dialogwechsel, sehr direktes Gespräch)", 600),
    ("mittellang (ca. 8-12 Dialogwechsel, normale Gesprächslänge)", 1000),
    ("lang und ausführlich (ca. 14-20 Dialogwechsel, detaillierte Diskussion)", 1600)
]

## 4. Strukturiertes Ausgabeformat mit Pydantic

`TranscriptSchema` beschreibt, wie ein erzeugter Datensatz aussehen soll.

Er erwartet:

- `id`: eindeutige Dokument-ID
- `text`: vollständiger Gesprächsverlauf
- `source`: Herkunft des Datensatzes

Der Vorteil: Die LLM-Ausgabe wird nicht einfach ungeprüft übernommen. Sie wird zunächst in eine definierte Datenstruktur überführt.


In [11]:
# Structured Output Schema definieren
class TranscriptSchema(BaseModel):
    id: str
    text: str = Field(description="Der komplette Gesprächsverlauf zwischen Kunde und Agent")
    source: str = "nemo-curator-custom"

system_prompt = (
    "Du bist ein spezialisierter Data-Generator für Security- & Fraud-Detection-Modelle im Banking-Sektor.\n"
    "Deine Aufgabe ist es, realistisch klingende Transkripte von Telefonaten zwischen einem Anrufer und einem Bankmitarbeiter (Agent) zu erzeugen.\n\n"
    "WICHTIG:\n"
    "1. Verwende für den Kunden im Dialog den Namen, der im Prompt vorgegeben wird, und sprich ihn auch so an.\n"
    "2. Bei Fraud-Calls versucht der Anrufer, den Agenten durch Täuschung, Ausreden, Druck oder Manipulation zu unberechtigten Aktionen zu bewegen.\n"
    "3. Verwende im Text strikt die Sprecher-Präfixe 'Kunde:' bzw. den Namen und 'Agent:'.\n"
    "4. Antworte AUSSCHLIESSLICH mit einem validen JSON-Objekt ohne Markdown-Formatierung:\n"
    "{\n"
    '  "id": "doc-XXX",\n'
    '  "text": "Kunde: ... \\nAgent: ... \\nKunde: ...",\n'
    '  "source": "nemo-curator-custom"\n'
    "}"
)

## 5. Verbindungstest zum LLM

Bevor 100 Generierungen gestartet werden, prüft `test_llm_connection()`, ob der konfigurierte LLM-Endpunkt erreichbar ist.

Dazu wird eine sehr kleine Anfrage gestellt:

> Antworte nur mit 'OK'

Kommt eine Antwort zurück, wird die Verbindung als erfolgreich betrachtet. Bei einem Fehler beendet das Skript den Start über `SystemExit(1)`.

Das ist sinnvoll, weil ein früher Verbindungstest verhindert, dass erst viele asynchrone Tasks gestartet werden und anschließend alle wegen eines Infrastrukturfehlers scheitern.


In [12]:
async def test_llm_connection():
    """Testet vorab über den NeMo Curator Client, ob der LLM-Container erreichbar ist."""
    print(f"🔍 Teste Verbindung zum LLM über NeMo Curator unter {NIM_BASE_URL} (Modell: {GEN_MODEL})...")
    try:
        response = curator_openai_client.query_model(
            model=GEN_MODEL,
            messages=[{"role": "user", "content": "Antworte nur mit 'OK'"}],
            max_tokens=10
        )
        answer = response[0].strip()
        print(f"✅ Verbindung erfolgreich! Test-Antwort vom Modell: '{answer}'")
    except Exception as e:
        print(f"❌ Verbindungstest zum LLM-Container fehlgeschlagen: {e}")
        raise SystemExit(1)

## 6. LLM-as-a-Judge

`evaluate_sample_quality()` verwendet das LLM ein zweites Mal – diesmal nicht zum Generieren, sondern zum Bewerten.

Das Modell soll einen Score von **1 bis 5** vergeben:

- 1 = unrealistisch / unbrauchbar
- 5 = sehr realistisch und gut geeignet

Anschließend wird die Antwort als JSON eingelesen.

Interessant ist hier die Kombination aus `asyncio` und `run_in_executor()`: Der verwendete Curator-Aufruf ist offenbar synchron, deshalb wird er in einen Executor ausgelagert, damit der Event Loop nicht unnötig blockiert.

Bei einem Fehler wird `0` zurückgegeben. Damit fällt der Datensatz automatisch durch das Qualitäts-Gate.


In [13]:
async def evaluate_sample_quality(text: str) -> int:
    """LLM-as-a-Judge: Bewertet das generierte Transkript mit dem LLM."""
    eval_prompt = (
        "Bewerte dieses Bank-Transkript auf einer Skala von 1-5 hinsichtlich Realismus "
        "und Eignung für ein Fraud-Detection-Training (SFT).\n"
        "1 = Müll/unrealistisch, 5 = Perfekt.\n"
        f"Transkript: {text}\n"
        "Antworte NUR mit einem JSON: {\"score\": int}"
    )
    
    loop = asyncio.get_running_loop()
    try:
        response = await loop.run_in_executor(
            None,
            lambda: curator_openai_client.query_model(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": eval_prompt}],
                max_tokens=50
            )
        )
        clean_json = re.sub(r"^```(?:json)?\s*|\s*```$", "", response[0].strip())
        data = json.loads(clean_json)
        return int(data.get("score", 0))
    except Exception:
        return 0

## 7. Einzelnen Datensatz erzeugen

`generate_single_sample()` ist der zentrale Teil des Programms.

### Schritt 1: Fraud oder legitim?

```python
is_fraud = (index % 2 != 0)
```

Ungerade IDs werden als `fraud`, gerade IDs als `legitimate` behandelt. Dadurch entsteht bei 100 Samples grundsätzlich eine 50/50-Verteilung der **Generierungsaufgaben**.

### Schritt 2: Zufällige Parameter

Für jeden Datensatz werden zufällig ausgewählt:

- Tonlage
- Gesprächslänge
- Kundenname
- passendes Szenario

### Schritt 3: Prompt bauen

Aus diesen Bausteinen entsteht der eigentliche Benutzer-Prompt.

### Schritt 4: Parallelitätsbegrenzung

```python
async with semaphore:
```

stellt sicher, dass höchstens `CONCURRENCY_LIMIT` Aufgaben gleichzeitig den geschützten Bereich ausführen.

### Schritt 5: Bis zu drei Versuche

Die Generierung läuft in einer Schleife mit maximal drei Versuchen. Bei Fehlern wird mit wachsender Wartezeit erneut versucht.


In [14]:
async def generate_single_sample(index: int, f_data, f_bench, rails_app: LLMRails):
    global completed_counter
    doc_id = f"doc-{index:05d}"
    is_fraud = (index % 2 != 0)
    label = "fraud" if is_fraud else "legitimate"
    tone = random.choice(TONES)
    length_desc, max_tokens_limit = random.choice(LENGTH_PROMPTS)
    customer_name = random.choice(KUNDEN_NAMEN)
    
    if is_fraud:
        scenario = random.choice(FRAUD_SCENARIOS)
        user_prompt = (
            f"Generiere ein BETRUGSGESPRAECH (Fraud Call / Social Engineering Inbound Call).\n"
            f"Name des Kunden: {customer_name}.\n"
            f"Szenario: Der Anrufer gibt sich als dieser Kunde aus und versucht den Bankmitarbeiter zu überlisten. Details: {scenario}\n"
            f"Stimmung des Anrufers: {tone}.\n"
            f"Gesprächslänge: {length_desc}."
        )
    else:
        scenario = random.choice(LEGITIMATE_SCENARIOS)
        user_prompt = (
            f"Generiere ein LEGITIMES, normales Kundengespräch am Telefon (Legitimate Call).\n"
            f"Name des Kunden: {customer_name}.\n"
            f"Szenario: Der echte Kunde ruft beim Kundenservice der Bank an. Details: {scenario}\n"
            f"Stimmung des Kunden: {tone}.\n"
            f"Gesprächslänge: {length_desc}."
        )

    async with semaphore:
        for attempt in range(3):
            try:
                temp = round(random.uniform(0.7, 0.95), 2)
                
                # Generierung über NeMo Guardrails (mit lokalem NIM-Modell)
                messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ]
                
                # Guardrails-Aufruf (asynchron)
                rails_response = await rails_app.generate_async(messages=messages)
                
                # Extraktion der Antwort je nach Rückgabeformat von LLMRails
                if isinstance(rails_response, dict):
                    raw_content = rails_response.get("content", str(rails_response))
                else:
                    raw_content = str(rails_response)

                clean_text = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_content, flags=re.MULTILINE).strip()
                clean_text_fixed = re.sub(r'\\(?!["\\/bfnrt]|u[0-9a-fA-F]{4})', r'\\\\', clean_text)
                
                try:
                    raw_json = json.loads(clean_text_fixed)
                except json.JSONDecodeError:
                    raw_json = {"id": doc_id, "text": clean_text.replace('\\', '/'), "source": "nemo-curator-custom"}

                # Validierung über Pydantic Schema
                data = TranscriptSchema(
                    id=doc_id,
                    text=raw_json.get("text", raw_content),
                    source="nemo-curator-custom"
                )

                # LLM-as-a-Judge Qualitätsprüfung
                score = await evaluate_sample_quality(data.text)

                if score >= 4:  # Qualitäts-Gate
                    # Thread-sicheres Schreiben mit asyncio.Lock
                    async with file_lock:
                        f_data.write(data.model_dump_json() + "\n")
                        f_data.flush()
                        
                        f_bench.write(json.dumps({"id": doc_id, "label": label, "score": score}, ensure_ascii=False) + "\n")
                        f_bench.flush()

                    completed_counter += 1
                    if completed_counter % 50 == 0 or completed_counter == NUM_SAMPLES:
                        print(f"⏳ Fortschritt (Curated & Judged): [{completed_counter}/{NUM_SAMPLES}] ({completed_counter/NUM_SAMPLES*100:.1f}%)")
                    return
                else:
                    if attempt == 2:
                        return

            except Exception as e:
                if attempt == 2:
                    print(f"❌ Guardrails-Fehler bei {doc_id} nach 3 Versuchen: {e}")
                await asyncio.sleep(1 * (attempt + 1))

## 8. Guardrails und LLM-Aufruf

Der Prompt wird als `system` und `user` Message an `rails_app.generate_async()` übergeben.

Die Idee von Guardrails ist, die Interaktion mit dem Modell kontrollierter zu machen. Im vorliegenden Code wird die Guardrails-Konfiguration später in `main()` mit dem lokalen OpenAI-kompatiblen Endpoint verbunden.

Danach versucht der Code, verschiedene Rückgabeformen von `LLMRails` zu behandeln:

```python
if isinstance(rails_response, dict):
    ...
else:
    ...
```

Anschließend werden mögliche Markdown-Codeblöcke entfernt und problematische Backslashes bereinigt, bevor `json.loads()` versucht, die Antwort als JSON zu interpretieren.


## 9. Fehlerbehandlung bei ungültigem JSON

Wenn das LLM kein gültiges JSON liefert, greift ein Fallback:

```python
raw_json = {
    "id": doc_id,
    "text": clean_text.replace('\\', '/'),
    "source": "nemo-curator-custom"
}
```

Das ist ein wichtiger Robustheitsmechanismus: Eine nicht perfekte LLM-Ausgabe führt nicht unmittelbar zum Abbruch des gesamten Generators.

Danach wird `TranscriptSchema` verwendet, um die resultierenden Felder zu validieren.


## 10. Qualitäts-Gate und Speichern

Nach der Pydantic-Validierung wird der Text durch `evaluate_sample_quality()` bewertet.

Nur wenn

```python
score >= 4
```

gilt, wird der Datensatz gespeichert.

Dabei entstehen zwei JSONL-Dateien:

**Transkriptdatei**
```text
{"id": "...", "text": "...", "source": "..."}
```

**Benchmarkdatei**
```text
{"id": "...", "label": "fraud", "score": 5}
```

`asyncio.Lock()` schützt den Schreibvorgang. Das ist wichtig, weil mehrere asynchrone Tasks gleichzeitig fertig werden können und gleichzeitig auf dieselben Dateien zugreifen.


## 11. Die Hauptfunktion

`main()` verbindet alle Bausteine.

1. Verbindung zum LLM testen
2. `RailsConfig` für NeMo Guardrails erzeugen
3. `LLMRails` initialisieren
4. 100 Generierungs-Tasks erstellen
5. Dateien öffnen
6. alle Tasks mit `asyncio.gather()` parallel ausführen
7. Abschlussmeldung ausgeben

Der Ausdruck

```python
tasks = [
    generate_single_sample(i, f_data, f_bench, rails_app)
    for i in range(1, NUM_SAMPLES + 1)
]
```

erzeugt die 100 Coroutine-Objekte.

`await asyncio.gather(*tasks)` wartet anschließend darauf, dass alle Aufgaben abgeschlossen sind.


In [ ]:
async def main():
    await test_llm_connection()

    # NeMo Guardrails Konfiguration für den lokalen NIM-Server laden
    config = RailsConfig.from_content(
        colang_content="",
        yaml_content=f"""
models:
  - type: main
    engine: openai
    model: {GEN_MODEL}
    parameters:
      base_url: {NIM_BASE_URL}
      api_key: not-needed
        """
    )
    rails_app = LLMRails(config)

    print(f"\n🚀 Starte Generierung mit Guardrails & Curator von bis zu {NUM_SAMPLES} Datensätzen (Max Concurrency: {CONCURRENCY_LIMIT})...")
    
    with open(OUTPUT_DATA_FILE, "a", encoding="utf-8") as f_data, \
         open(OUTPUT_BENCHMARK_FILE, "a", encoding="utf-8") as f_bench:
        
        tasks = [generate_single_sample(i, f_data, f_bench, rails_app) for i in range(1, NUM_SAMPLES + 1)]
        await asyncio.gather(*tasks)

    print(f"\n✅ Fertig! Validierte Datensätze wurden in {OUTPUT_DATA_FILE} und {OUTPUT_BENCHMARK_FILE} gespeichert.")

if __name__ == "__main__":
    await main()

🔍 Teste Verbindung zum LLM über NeMo Curator unter http://172.17.0.1:8800/v1 (Modell: meta/llama-3.1-8b-instruct)...
✅ Verbindung erfolgreich! Test-Antwort vom Modell: 'OK'

🚀 Starte Generierung mit Guardrails & Curator von bis zu 5000 Datensätzen (Max Concurrency: 8)...
⏳ Fortschritt (Curated & Judged): [50/5000] (1.0%)
⏳ Fortschritt (Curated & Judged): [100/5000] (2.0%)
⏳ Fortschritt (Curated & Judged): [150/5000] (3.0%)
⏳ Fortschritt (Curated & Judged): [200/5000] (4.0%)
⏳ Fortschritt (Curated & Judged): [250/5000] (5.0%)
⏳ Fortschritt (Curated & Judged): [300/5000] (6.0%)
⏳ Fortschritt (Curated & Judged): [350/5000] (7.0%)
⏳ Fortschritt (Curated & Judged): [400/5000] (8.0%)
⏳ Fortschritt (Curated & Judged): [450/5000] (9.0%)
⏳ Fortschritt (Curated & Judged): [500/5000] (10.0%)


ERROR:nemoguardrails.rails.llm.llmrails:Error in generate_async: Error invoking LLM (model=meta/llama-3.1-8b-instruct, provider=openai, endpoint=http://172.17.0.1:8800/v1): (model=meta/llama-3.1-8b-instruct, provider=openai, endpoint=http://172.17.0.1:8800/v1) Request timed out:
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/httpx/_transports/default.py", line 72, in map_httpcore_exceptions
    yield
  File "/usr/local/lib/python3.10/dist-packages/httpx/_transports/default.py", line 377, in handle_async_request
    resp = await self._pool.handle_async_request(req)
  File "/usr/local/lib/python3.10/dist-packages/httpcore/_async/connection_pool.py", line 216, in handle_async_request
    raise exc from None
  File "/usr/local/lib/python3.10/dist-packages/httpcore/_async/connection_pool.py", line 196, in handle_async_request
    response = await connection.handle_async_request(
  File "/usr/local/lib/python3.10/dist-packages/httpcore/_async/connection.py

⏳ Fortschritt (Curated & Judged): [550/5000] (11.0%)
⏳ Fortschritt (Curated & Judged): [600/5000] (12.0%)
⏳ Fortschritt (Curated & Judged): [650/5000] (13.0%)


ERROR:nemoguardrails.rails.llm.llmrails:Error in generate_async: Error invoking LLM (model=meta/llama-3.1-8b-instruct, provider=openai, endpoint=http://172.17.0.1:8800/v1): (model=meta/llama-3.1-8b-instruct, provider=openai, endpoint=http://172.17.0.1:8800/v1) Request timed out:
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/httpx/_transports/default.py", line 72, in map_httpcore_exceptions
    yield
  File "/usr/local/lib/python3.10/dist-packages/httpx/_transports/default.py", line 377, in handle_async_request
    resp = await self._pool.handle_async_request(req)
  File "/usr/local/lib/python3.10/dist-packages/httpcore/_async/connection_pool.py", line 216, in handle_async_request
    raise exc from None
  File "/usr/local/lib/python3.10/dist-packages/httpcore/_async/connection_pool.py", line 196, in handle_async_request
    response = await connection.handle_async_request(
  File "/usr/local/lib/python3.10/dist-packages/httpcore/_async/connection.py

⏳ Fortschritt (Curated & Judged): [700/5000] (14.0%)
⏳ Fortschritt (Curated & Judged): [750/5000] (15.0%)
⏳ Fortschritt (Curated & Judged): [800/5000] (16.0%)
⏳ Fortschritt (Curated & Judged): [1050/5000] (21.0%)


...

❌ Guardrails-Fehler bei doc-02532 nach 3 Versuchen: Error invoking LLM (model=meta/llama-3.1-8b-instruct, provider=openai, endpoint=http://172.17.0.1:8800/v1): [502] (model=meta/llama-3.1-8b-instruct, provider=openai, endpoint=http://172.17.0.1:8800/v1) <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>nginx/1.31.3</center>
</body>
</html>


ERROR:nemoguardrails.rails.llm.llmrails:Error in generate_async: Error invoking LLM (model=meta/llama-3.1-8b-instruct, provider=openai, endpoint=http://172.17.0.1:8800/v1): [502] (model=meta/llama-3.1-8b-instruct, provider=openai, endpoint=http://172.17.0.1:8800/v1) <html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>nginx/1.31.3</center>
</body>
</html>
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/nemoguardrails/actions/llm/utils.py", line 81, in llm_call
    response: LLMResponse = await model.generate_async(chat_prompt, stop=stop, **(llm_params or {}))
  File "/usr/local/lib/python3.10/dist-packages/nemoguardrails/llm/models/openai_chat.py", line 117, in generate_async
    raise self._enrich(exc)
  File "/usr/local/lib/python3.10/dist-packages/nemoguardrails/llm/models/openai_chat.py", line 115, in generate_async
    response = await self._client.chat_completion(self._model, messages, *

⏳ Fortschritt (Curated & Judged): [2200/5000] (44.0%)
⏳ Fortschritt (Curated & Judged): [2250/5000] (45.0%)


ERROR:nemoguardrails.rails.llm.llmrails:Error in generate_async: Error invoking LLM (model=meta/llama-3.1-8b-instruct, provider=openai, endpoint=http://172.17.0.1:8800/v1): (model=meta/llama-3.1-8b-instruct, provider=openai, endpoint=http://172.17.0.1:8800/v1) Request timed out:
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/httpx/_transports/default.py", line 72, in map_httpcore_exceptions
    yield
  File "/usr/local/lib/python3.10/dist-packages/httpx/_transports/default.py", line 377, in handle_async_request
    resp = await self._pool.handle_async_request(req)
  File "/usr/local/lib/python3.10/dist-packages/httpcore/_async/connection_pool.py", line 216, in handle_async_request
    raise exc from None
  File "/usr/local/lib/python3.10/dist-packages/httpcore/_async/connection_pool.py", line 196, in handle_async_request
    response = await connection.handle_async_request(
  File "/usr/local/lib/python3.10/dist-packages/httpcore/_async/connection.py

⏳ Fortschritt (Curated & Judged): [2300/5000] (46.0%)


## 12. Warum `asyncio`?

Ohne Parallelisierung würde das Programm die LLM-Anfragen nacheinander abarbeiten.

Mit `asyncio` können mehrere Anfragen gleichzeitig auf Antworten warten. Der `Semaphore` begrenzt dabei die Anzahl gleichzeitig aktiver Aufgaben.

Vereinfacht:

```text
100 Aufgaben
    │
    ▼
Semaphore (max. 8 gleichzeitig)
    │
    ├── Aufgabe 1 → LLM
    ├── Aufgabe 2 → LLM
    ├── ...
    └── Aufgabe 8 → LLM
             │
             ▼
       Qualitätsprüfung
             │
        ┌────┴────┐
        │         │
      Score ≥4   Score <4
        │         │
      speichern  verwerfen
```

Das ist besonders sinnvoll bei LLM-Aufrufen, weil viel Zeit auf I/O bzw. das Warten auf Antworten entfällt.


## 13. Gesamtarchitektur

Der Datenfluss des Skripts lautet:

**Szenario + Name + Ton + Länge**

→ Prompt

→ **NeMo Guardrails**

→ **lokales NIM / LLM**

→ JSON-Bereinigung

→ **Pydantic-Validierung**

→ **LLM-as-a-Judge**

→ Qualitäts-Gate (`score >= 4`)

→ **JSONL-Datensatz + Benchmark**

Damit ist das Skript nicht nur ein einfacher Textgenerator, sondern eine kleine Data-Curation-Pipeline.


## 14. Wichtige Punkte beim Betrieb

### Umgebung

Das Skript erwartet einen erreichbaren OpenAI-kompatiblen NIM-Endpunkt unter der konfigurierten URL.

### Ausgabepfade

Die beiden Pfade sind fest im Skript hinterlegt:

- `/data/nemo-fraud-detection/data/raw/transcripts.jsonl`
- `/data/nemo-fraud-detection/data/raw/fraud_call_benchmark_curator.jsonl`

Diese Verzeichnisse müssen vorhanden und beschreibbar sein.

### Anzahl der Ergebnisse

`NUM_SAMPLES = 100` bedeutet **bis zu** 100 gespeicherte Datensätze. Da nur Scores ≥ 4 gespeichert werden, kann die tatsächlich gespeicherte Anzahl kleiner sein.

### Label-Verteilung

Die Fraud/Legitimate-Auswahl wird über den Index gesteuert. Das Label selbst wird also nicht vom Judge bestimmt, sondern vom Generator festgelegt.


## 15. Kleine Übung

Bevor du das komplette Skript ausführst, kannst du diese Fragen beantworten:

1. Was bewirkt `asyncio.Semaphore(8)`?
2. Warum wird `Pydantic` nach der LLM-Generierung verwendet?
3. Was passiert bei einem Qualitäts-Score von 3?
4. Warum wird `asyncio.Lock()` beim Schreiben verwendet?
5. Wie wird die Fraud/Legitimate-Verteilung im aktuellen Code erzeugt?
6. Was ist die Aufgabe des LLM-as-a-Judge?
7. Warum kann die Zahl der gespeicherten Datensätze kleiner als `NUM_SAMPLES` sein?


## Fazit

Der Code implementiert eine **asynchrone synthetische Datenpipeline für Fraud-Detection**.

Besonders wichtig sind vier Ebenen:

1. **Generierung:** LLM erzeugt realistische Telefontranskripte.
2. **Strukturierung:** JSON und Pydantic bringen die Ergebnisse in ein definiertes Format.
3. **Qualitätssicherung:** Ein Judge-LLM bewertet die erzeugten Texte.
4. **Parallelisierung:** `asyncio` ermöglicht eine effiziente Verarbeitung vieler Samples.

Das resultierende JSONL-Format eignet sich anschließend als Grundlage für weitere Datenaufbereitung bzw. SFT-Experimente.
